# ReServe NF4 Profiling

Run these cells in order on Kaggle with one Tesla T4 visible as GPU 0. The profiler reuses a cached Qwen checkpoint when available and downloads a pinned checkpoint if needed. Internet is needed for dependency installation and any missing model files.

The low mode uses bitsandbytes NF4 weights, FP16 computation, and a verified FP16 KV cache. It profiles fixed modes for the ReServe simulator.

The grid, order, input token, 10 warm-ups, and 50 measured trials match the uploaded FP16 run `fp16_20261126T204405_157396Z`: batch sizes 1, 2, 4, 8 and lengths 256, 512, 1024, 2048. Length means prompt tokens for prefill and existing cached tokens for decode. Each decode trial gets a fresh cache.

The GPU, PyTorch, Transformers and CUDA runtime are checked against that FP16 run. No earlier notebook helper functions are required. The profiler releases an existing global `model` before loading NF4.


## Install dependencies

The Transformers version is pinned to your FP16 run. If the profiler reports that package availability is stale after installation, restart only the Python kernel and run the profiler cell again; it has all its definitions.


In [ ]:
%pip install -q "bitsandbytes==0.50.2" "accelerate>=1.1" "transformers==5.20.2"


## Run the profiling sweep

This cell first verifies real 4-bit linear layers, finite prefill/decode outputs, FP16 KV tensors and 28 KiB of raw KV per cached token. It then collects 32 case summaries and up to 1,600 timed samples. Warm-ups, decode-cache preparation, token selection and file writing are excluded from timing. Host launch overhead is included.

Results are saved after each case to a new folder under `/kaggle/working/reserve_profiles/`. Out-of-memory cases retain their completed samples and are labeled `oom`; other errors stop the sweep after checkpointing. Use `status == "ok"` rows for calibration.


In [ ]:
"""ReServe fixed-mode NF4 profiling; paste/run in a Kaggle GPU notebook."""
import gc
import json
import platform
import statistics
import time
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import torch
import transformers
import bitsandbytes as bnb
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.errors import LocalEntryNotFoundError
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from transformers.utils import is_bitsandbytes_available, is_accelerate_available

# Reference settings read from Daniel's uploaded FP16 metadata.json.
REFERENCE = {
    "run_id": "fp16_20261126T204405_157396Z",
    "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
    "model_revision": None,
    "gpu": "Tesla T4",
    "torch": "2.11.0+cu130",
    "transformers": "5.20.2",
    "cuda_runtime": "13.0",
    "input_token_id": 14990,
    "grid_seed": 42,
    "case_order": [
        [2, 2048], [4, 512], [2, 512], [2, 1024],
        [8, 1024], [4, 1024], [8, 256], [4, 256],
        [1, 512], [1, 1024], [8, 512], [8, 2048],
        [2, 256], [4, 2048], [1, 256], [1, 2048],
    ],
}
MODE = "bnb_nf4_fp16_compute_kv"
WARMUP_RUNS = 10
MEASURED_RUNS = 50
DEVICE = torch.device("cuda:0")
OUTPUT_ROOT = Path("/kaggle/working/reserve_profiles")


def summarize(samples):
    names = ["median_ms", "mean_ms", "std_ms", "q1_ms", "q3_ms", "iqr_ms",
             "min_ms", "max_ms", "cv_pct", "peak_allocated_mib", "peak_reserved_mib"]
    result = {name: None for name in names}
    result["measured_runs"] = len(samples)
    if not samples:
        return result
    values = [sample["latency_ms"] for sample in samples]
    q1, _, q3 = statistics.quantiles(values, n=4, method="inclusive") if len(values) > 1 else [values[0]] * 3
    mean = statistics.mean(values)
    std = statistics.stdev(values) if len(values) > 1 else None
    result.update({
        "median_ms": statistics.median(values), "mean_ms": mean, "std_ms": std,
        "q1_ms": q1, "q3_ms": q3, "iqr_ms": q3 - q1,
        "min_ms": min(values), "max_ms": max(values),
        "cv_pct": 100 * std / mean if std is not None and mean else None,
        "peak_allocated_mib": max(s["peak_allocated_mib"] for s in samples),
        "peak_reserved_mib": max(s["peak_reserved_mib"] for s in samples),
    })
    return result


@torch.inference_mode()
def measure_once(model, phase, ids, prompt_mask, decode_mask, device):
    prefix = cache = next_token = output = None
    try:
        if phase == "decode":
            # A NEW cache for every trial: context length never accumulates.
            # Both prefill and token selection are outside the decode timer.
            prefix = model(input_ids=ids, attention_mask=prompt_mask,
                           use_cache=True, logits_to_keep=1)
            cache = prefix.past_key_values
            assert cache is not None
            next_token = prefix.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            assert tuple(next_token.shape) == (ids.shape[0], 1)
            prefix = None

        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
        start = time.perf_counter()
        if phase == "prefill":
            output = model(input_ids=ids, attention_mask=prompt_mask,
                           use_cache=True, logits_to_keep=1)
        else:
            output = model(input_ids=next_token, attention_mask=decode_mask,
                           past_key_values=cache, use_cache=True, logits_to_keep=1)
        torch.cuda.synchronize(device)
        elapsed_ms = (time.perf_counter() - start) * 1000
        assert output.logits.shape[0] == ids.shape[0]
        return {
            "latency_ms": elapsed_ms,
            "peak_allocated_mib": torch.cuda.max_memory_allocated(device) / 1024**2,
            "peak_reserved_mib": torch.cuda.max_memory_reserved(device) / 1024**2,
        }
    finally:
        prefix = cache = next_token = output = None


def checkpoint(directory, summaries, raw_rows, metadata):
    raw_columns = ["run_id", "mode", "phase", "batch_size", "length_tokens", "trial",
                   "latency_ms", "peak_allocated_mib", "peak_reserved_mib"]
    outputs = {
        "summary.csv": pd.DataFrame(summaries).sort_values(["phase", "batch_size", "length_tokens"]),
        "raw_timings.csv": pd.DataFrame(raw_rows, columns=raw_columns),
    }
    for name, frame in outputs.items():
        temporary = directory / (name + ".tmp")
        frame.to_csv(temporary, index=False)
        temporary.replace(directory / name)
    temporary = directory / "metadata.json.tmp"
    temporary.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    temporary.replace(directory / "metadata.json")


def sweep(model, directory, metadata, cases, token_id, device, warmups, repeats):
    summaries, raw_rows = [], []
    for batch_size, length in cases:
        for phase in ("prefill", "decode"):
            case = {"run_id": metadata["run_id"], "mode": MODE, "phase": phase,
                    "batch_size": batch_size, "length_tokens": length}
            samples, status, error = [], "ok", ""
            completed_warmups = 0
            fatal_error = None
            ids = prompt_mask = decode_mask = None
            print(f"Running {phase}: batch={batch_size}, length={length}", flush=True)
            try:
                ids = torch.full((batch_size, length), token_id, dtype=torch.long, device=device)
                prompt_mask = torch.ones_like(ids)
                decode_mask = torch.ones((batch_size, length + 1), dtype=torch.long, device=device)
                for _ in range(warmups):
                    measure_once(model, phase, ids, prompt_mask, decode_mask, device)
                    completed_warmups += 1
                for trial in range(repeats):
                    sample = measure_once(model, phase, ids, prompt_mask, decode_mask, device)
                    samples.append(sample)
                    raw_rows.append({**case, "trial": trial + 1, **sample})
            except torch.cuda.OutOfMemoryError as exc:
                status, error = "oom", str(exc).splitlines()[0]
            except KeyboardInterrupt as exc:
                status, error, fatal_error = "interrupted", "User interrupted this case", exc
            except Exception as exc:
                status, error, fatal_error = "error", f"{type(exc).__name__}: {exc}", exc
            finally:
                ids = prompt_mask = decode_mask = None
                gc.collect()
                torch.cuda.empty_cache()

            row = {**case, "status": status, "error": error,
                   "warmups_completed": completed_warmups, **summarize(samples)}
            summaries.append(row)
            metadata["cases_recorded"] = len(summaries)
            metadata["run_status"] = status if fatal_error is not None else "running"
            checkpoint(directory, summaries, raw_rows, metadata)
            if samples:
                print(f"{status} | n={len(samples)} | median={row['median_ms']:.2f} ms | "
                      f"IQR={row['iqr_ms']:.2f} ms", flush=True)
            else:
                print(f"{status} | {error}", flush=True)
            if fatal_error is not None:
                raise fatal_error

    metadata["run_status"] = "completed_with_oom" if any(r["status"] == "oom" for r in summaries) else "completed"
    metadata["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
    checkpoint(directory, summaries, raw_rows, metadata)
    return pd.DataFrame(summaries).sort_values(["phase", "batch_size", "length_tokens"])


def run_nf4_profile():
    assert torch.cuda.is_available(), "Enable the Kaggle GPU first."
    assert torch.cuda.get_device_name(DEVICE) == REFERENCE["gpu"], "GPU differs from the FP16 run."
    assert torch.__version__ == REFERENCE["torch"], "PyTorch differs from the FP16 run."
    assert transformers.__version__ == REFERENCE["transformers"], "Transformers differs from the FP16 run."
    assert torch.version.cuda == REFERENCE["cuda_runtime"], "CUDA runtime differs from the FP16 run."
    assert MEASURED_RUNS >= 2 and WARMUP_RUNS >= 0
    # Some notebook sessions cache package-availability checks before pip runs.
    for available in (is_bitsandbytes_available, is_accelerate_available):
        clear = getattr(available, "cache_clear", None)
        if clear is not None:
            clear()
        if not available():
            raise RuntimeError(
                "A quantization dependency is unavailable to Transformers. "
                "After the installation cell succeeds, restart the Python kernel "
                "and rerun this self-contained profiling cell."
            )

    # Prefer the same cached checkpoint; resolve/pin a Hub commit if it is absent.
    try:
        source = str(snapshot_download(REFERENCE["model_id"], local_files_only=True))
        revision, checkpoint_source = Path(source).name, "cached_main_snapshot"
        load_options = {"local_files_only": True}
    except LocalEntryNotFoundError:
        revision = HfApi().model_info(REFERENCE["model_id"]).sha
        source, checkpoint_source = REFERENCE["model_id"], "hub_resolved_commit"
        load_options = {"revision": revision}

    config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16,
                               bnb_4bit_use_double_quant=False)
    gc.collect()
    torch.cuda.empty_cache()
    allocated_before_load = torch.cuda.memory_allocated(DEVICE) / 1024**2
    model = AutoModelForCausalLM.from_pretrained(
        source, quantization_config=config, dtype=torch.float16,
        device_map={"": 0}, attn_implementation="sdpa", **load_options,
    ).eval()
    tokenizer = AutoTokenizer.from_pretrained(source, **load_options)
    quantized_layers = sum(isinstance(m, bnb.nn.Linear4bit) for m in model.modules())
    assert quantized_layers > 0, "No NF4 linear layers were loaded."
    assert all(p.device == DEVICE for p in model.parameters()), "All weights must be on GPU 0."
    assert tokenizer.encode("hello", add_special_tokens=False)[0] == REFERENCE["input_token_id"]
    assert max(length for _, length in REFERENCE["case_order"]) + 1 <= model.config.max_position_embeddings

    inputs = tokenizer("Explain a KV cache briefly.", return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        prefill = model(**inputs, use_cache=True, logits_to_keep=1)
        cache = prefill.past_key_values
        kv_dtypes = {str(t.dtype) for layer in cache.layers for t in (layer.keys, layer.values)}
        assert kv_dtypes == {"torch.float16"}, f"Expected FP16 KV, got {kv_dtypes}"
        kv_bytes = sum(t.numel() * t.element_size() for layer in cache.layers for t in (layer.keys, layer.values))
        kv_bytes_per_token = kv_bytes // inputs["input_ids"].shape[1]
        assert kv_bytes_per_token == 28672, f"Unexpected KV footprint: {kv_bytes_per_token} bytes/token"
        next_token = prefill.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        mask = torch.ones((1, inputs["input_ids"].shape[1] + 1), dtype=torch.long, device=DEVICE)
        decoded = model(input_ids=next_token, attention_mask=mask,
                        past_key_values=cache, use_cache=True, logits_to_keep=1)
        assert torch.isfinite(prefill.logits).all().item()
        assert torch.isfinite(decoded.logits).all().item()
    torch.cuda.synchronize(DEVICE)
    del inputs, prefill, cache, next_token, mask, decoded
    gc.collect()
    torch.cuda.empty_cache()
    baseline_mib = torch.cuda.memory_allocated(DEVICE) / 1024**2

    run_id = MODE + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    directory = OUTPUT_ROOT / run_id
    directory.mkdir(parents=True, exist_ok=False)
    metadata = {
        "run_id": run_id, "reference_fp16_run_id": REFERENCE["run_id"],
        "model_id": REFERENCE["model_id"], "model_revision": revision,
        "checkpoint_source": checkpoint_source,
        "reference_fp16_model_revision": REFERENCE["model_revision"],
        "revision_note": "NF4 revision is recorded; FP16 revision was unrecorded, so exact cross-mode revision identity is not retrospectively verified.",
        "mode": MODE, "python": platform.python_version(),
        "torch": torch.__version__, "transformers": transformers.__version__,
        "cuda_runtime": torch.version.cuda, "bitsandbytes": version("bitsandbytes"),
        "accelerate": version("accelerate"), "gpu": torch.cuda.get_device_name(DEVICE),
        "gpu_total_bytes": torch.cuda.get_device_properties(DEVICE).total_memory,
        "attention_implementation_requested": "sdpa", "warmup_runs": WARMUP_RUNS,
        "measured_runs": MEASURED_RUNS, "grid_seed": REFERENCE["grid_seed"],
        "case_order": REFERENCE["case_order"], "input_token_id": REFERENCE["input_token_id"],
        "quantization_config": config.to_dict(), "quantized_linear_layers": quantized_layers,
        "kv_dtype_verified": "float16", "kv_bytes_per_token_verified": kv_bytes_per_token,
        "allocated_before_nf4_load_mib": allocated_before_load, "baseline_allocated_mib": baseline_mib,
        "model_config": model.config.to_dict(),
        "timing": "synchronized forward-pass wall time including host launch overhead",
        "decode_setup": "fresh prefill cache and argmax prepared outside the decode timer",
        "memory": "per-trial PyTorch peaks; decode includes its pre-existing cache",
        "coverage": "homogeneous fixed NF4 batches; no mixed-batch or live-switch measurements",
        "expected_cases": len(REFERENCE["case_order"]) * 2, "run_status": "running",
    }
    (directory / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"NF4 linear layers: {quantized_layers}; KV: FP16, {kv_bytes_per_token / 1024:.1f} KiB/token")
    print(f"Allocated after loading/checks: {baseline_mib:.1f} MiB; before load: {allocated_before_load:.1f} MiB")
    print(f"32 cases, {WARMUP_RUNS} warm-ups + {MEASURED_RUNS} measured trials each")
    print(f"Saving to: {directory}", flush=True)
    frame = sweep(model, directory, metadata, REFERENCE["case_order"],
                  REFERENCE["input_token_id"], DEVICE, WARMUP_RUNS, MEASURED_RUNS)
    return frame, directory


# Release the earlier model if this code is pasted into the existing notebook.
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

nf4_summary_df, nf4_output_dir = run_nf4_profile()
display(nf4_summary_df.round(3))
print(f"Saved summary.csv, raw_timings.csv and metadata.json in: {nf4_output_dir}")


## Optional download bundle

After the sweep finishes, this cell creates a ZIP containing the three output files. Download it from Kaggle’s output files.


In [ ]:
import shutil

zip_path = shutil.make_archive(
    str(nf4_output_dir), "zip", root_dir=nf4_output_dir
)
print("Download:", zip_path)


## Reading the results

- `summary.csv`: sample count, median, mean, sample standard deviation, quartiles, IQR, min/max, CV, PyTorch memory peaks and case status.
- `raw_timings.csv`: each measured trial, with its phase, batch size, context length and memory peaks.
- `metadata.json`: software, hardware, resolved checkpoint, NF4 settings, verified KV dtype/footprint, reference FP16 run and completion status.

Memory peaks are process-level PyTorch allocated/reserved bytes, not a direct measure of memory reclaimable for restored weights. The original FP16 run did not record a checkpoint revision; the new metadata records that limitation without claiming retrospective identity. These profiles cover homogeneous batches and fixed precision, with mixed-batch execution and switching/recovery calibration to be handled separately.

The loop accounting, statistics, checkpoint writes, and failure handling were checked locally using a test model. Actual NF4 CUDA execution is checked by the initial prefill/decode test on your Kaggle GPU.

Reference documentation: [bitsandbytes installation](https://huggingface.co/docs/bitsandbytes/v0.50.2/en/installation), [Transformers bitsandbytes](https://huggingface.co/docs/transformers/main/en/quantization/bitsandbytes), [Transformers caching](https://huggingface.co/docs/transformers/v5.20.2/cache_explanation).
